In [11]:
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [12]:
import duckdb
from pathlib import Path
from typing import List
import os

In [13]:
# Store database at project root
DB_NAME = Path("/home/c-enjalbert/Documents/Github/MSPR/bloc_2/amazing/amazing.duckdb") 
# Go up one level from current directory to get to project root
data_folder = Path("..") / "data"
# For absolute certainty, you could use the absolute path
# data_folder = Path("/home/c-enjalbert/Documents/EPSI/MSPR/bloc_2/amazing/data")
con = duckdb.connect(str(DB_NAME))

In [14]:
def get_file_paths() -> List[str]:
    return sorted([str(p.resolve()) for p in data_folder.glob("*.csv")])

def load_new_files(con, file_paths: List[str]):
    print(f"Début du chargement de {len(file_paths)} fichier(s)...\n")
    for i, path in enumerate(file_paths, 1):
        filename = os.path.basename(path)

        print(f"[{i}/{len(file_paths)}] Vérification de {filename}...")
        already_loaded = con.execute(
            "SELECT 1 FROM loaded_files WHERE filename = ?", [filename]
        ).fetchone()

        if already_loaded:
            print(f"{filename} déjà chargé. Ignoré.\n")
            continue

        print(f"⬆Chargement de {filename} dans all_events...")
        con.execute(f"""
            INSERT INTO all_events
            SELECT * FROM read_csv_auto('{path}', AUTO_DETECT=TRUE, SAMPLE_SIZE=-1)
        """)
        con.execute("INSERT INTO loaded_files VALUES (?)", [filename])
        print(f"{filename} ajouté avec succès à la base.\n")

    print("Chargement terminé.\n")

def init_loaded_table(con):
    print("Initialisation de la table 'loaded_files'...")
    con.execute("""
        CREATE TABLE IF NOT EXISTS loaded_files (
            filename TEXT PRIMARY KEY
        );
    """)
    print("Table 'loaded_files' prête.\n")

def create_all_events_table(con):
    con.execute("""
        CREATE TABLE IF NOT EXISTS all_events (
            event_time TIMESTAMP,
            event_type TEXT,
            product_id TEXT,
            category_id TEXT,
            category_code TEXT,
            brand TEXT,
            price DOUBLE,
            user_id TEXT,
            user_session TEXT
        );
    """)

In [15]:


# Initialisation des tables
init_loaded_table(con)
create_all_events_table(con)

# Chargement des fichiers
files = get_file_paths()
load_new_files(con, files)

# Test génération de la table events_tables
create_all_events_table(con)



Initialisation de la table 'loaded_files'...
Table 'loaded_files' prête.

Début du chargement de 7 fichier(s)...

[1/7] Vérification de 2019-Dec.csv...
⬆Chargement de 2019-Dec.csv dans all_events...
2019-Dec.csv ajouté avec succès à la base.

[2/7] Vérification de 2019-Nov.csv...
⬆Chargement de 2019-Nov.csv dans all_events...
2019-Nov.csv ajouté avec succès à la base.

[3/7] Vérification de 2019-Oct.csv...
⬆Chargement de 2019-Oct.csv dans all_events...
2019-Oct.csv ajouté avec succès à la base.

[4/7] Vérification de 2020-Apr.csv...
⬆Chargement de 2020-Apr.csv dans all_events...
2020-Apr.csv ajouté avec succès à la base.

[5/7] Vérification de 2020-Feb.csv...
⬆Chargement de 2020-Feb.csv dans all_events...
2020-Feb.csv ajouté avec succès à la base.

[6/7] Vérification de 2020-Jan.csv...
⬆Chargement de 2020-Jan.csv dans all_events...
2020-Jan.csv ajouté avec succès à la base.

[7/7] Vérification de 2020-Mar.csv...
⬆Chargement de 2020-Mar.csv dans all_events...
2020-Mar.csv ajouté avec su

In [16]:
nb_users = con.execute("SELECT COUNT(*) FROM all_events").fetchone()[0]

print(f"Taille de la table all_events : {nb_users} logs")

df_purchase = con.execute("SELECT * FROM all_events WHERE user_id = '535135317' LIMIT 10").fetch_df()
print(df_purchase)


Taille de la table all_events : 411709736 logs
           event_time event_type product_id          category_id  \
0 2019-12-01 00:00:02   purchase   26400248  2053013553056579841   
1 2019-12-01 00:00:23       view   26400248  2053013553056579841   
2 2019-12-01 17:20:53       view   26400248  2053013553056579841   
3 2019-12-01 17:21:00       cart   26400248  2053013553056579841   
4 2019-12-01 17:21:10   purchase   26400248  2053013553056579841   
5 2019-12-01 17:21:51       view   26400248  2053013553056579841   
6 2019-12-02 10:23:14       view    1801881  2232732099754852875   
7 2019-12-22 10:00:19       view    1801881  2232732099754852875   
8 2019-12-22 10:01:22       view    4100151  2232732098228126185   
9 2019-12-22 10:02:48       view    1801881  2232732099754852875   

                   category_code    brand   price    user_id  \
0  computers.peripherals.printer      NaN  132.31  535135317   
1  computers.peripherals.printer      NaN  132.31  535135317   
2  computers

In [1]:
# Fermeture de la connexion DuckDB
con.close()

NameError: name 'con' is not defined